In [4]:
from molecules import CaOH, CaH

molecule = CaH.from_file(b_field_gauss=3.1, j_max=50)
print(len(molecule.state_df))
molecule.state_df

5202


,j,m,xi,spin_up,spin_down,zeeman_energy_khz,rotation_energy_ghz
0,0,-0.5,False,0.000000,1.000000,6.599509,0.0
1,0,0.5,True,1.000000,0.000000,-6.599509,0.0
2,1,-1.5,False,0.000000,1.000000,-0.874176,288.0
3,1,-0.5,False,0.380705,-0.924697,9.079865,288.0
4,1,0.5,False,0.260806,-0.965391,15.700762,288.0
...,...,...,...,...,...,...,...
5197,50,46.5,True,0.981403,0.191960,-71.142981,367200.0
5198,50,47.5,True,0.986095,0.166184,-68.086212,367200.0
5199,50,48.5,True,0.990758,0.135642,-65.029333,367200.0
5200,50,49.5,True,0.995393,0.095880,-61.972343,367200.0


In [5]:
print(len(molecule.transition_df))
molecule.transition_df

10201


,j,m1,xi1,m2,xi2,index1,index2,energy_diff,coupling
0,0,0.5,True,-0.5,False,1,0,13.199018,-0.000000
1,1,-0.5,False,-1.5,False,3,2,-9.954041,0.184939
2,1,0.5,False,-0.5,False,4,3,-6.620897,0.158681
3,1,0.5,False,-0.5,True,4,5,-23.734313,-0.121739
4,1,-0.5,True,-1.5,False,5,2,7.159374,-0.076141
...,...,...,...,...,...,...,...,...,...
10196,50,48.5,True,47.5,False,5199,5098,442.675649,0.005938
10197,50,49.5,True,48.5,True,5200,5199,-3.056989,0.094235
10198,50,49.5,True,48.5,False,5200,5099,442.989151,0.006258
10199,50,50.5,True,49.5,True,5201,5200,-3.057099,0.068335


In [6]:
frequency = 1e-2

detunings = 2 * np.pi * (frequency - molecule.transition_df["energy_diff"].to_numpy(dtype=float) * 1e-3)
print(detunings)
print(len(detunings))

[-0.02010003  0.12537494  0.10443217 ... -2.72055107  0.08204018
 -2.72251948]
10201


In [7]:
rabi_rate_mhz = 2*1e-2

omegas = rabi_rate_mhz * molecule.transition_df["coupling"].to_numpy(dtype=float)
print(omegas)
print(len(omegas))

[-0.          0.00369879  0.00317361 ...  0.00012516  0.0013667
  0.00013165]
10201


In [8]:
duration_us = 1e2

transition_exc_probs = omegas**2 / (omegas**2 + detunings**2) * np.sin(np.sqrt(omegas**2.0 + detunings**2.0) * duration_us / 2) ** 2
print(transition_exc_probs)
print(len(transition_exc_probs))

[0.00000000e+00 1.19255321e-07 7.01486805e-04 ... 1.37831090e-09
 1.86440550e-04 1.73385347e-09]
10201


In [9]:
rows = molecule.transition_df["index2"].to_numpy(dtype=int)
print(rows)
print(rows[rows == 34])


[   0    2    3 ... 5099 5200 5100]
[34 34]


In [10]:
states_index = molecule.transition_df["index1"].to_numpy(dtype=int)

print(len(states_index))
print(states_index)

10201
[   1    3    4 ... 5200 5201 5201]


In [11]:
state_exc_probs = np.zeros(len(molecule.state_df)) 

for i in range(len(molecule.transition_df)):
    state_exc_probs[states_index[i]] += transition_exc_probs[i]

print(state_exc_probs)

[0.         0.         0.         ... 0.00050564 0.00035471 0.00018644]


In [20]:
temperature = 300

rotational_energy_ghz = molecule.state_df["rotation_energy_ghz"].to_numpy()
state_distribution = np.exp(-h * rotational_energy_ghz * 1e9 / (k * temperature))

print(state_distribution)
print(len(state_distribution))

state_distribution /= np.sum(state_distribution)
print(state_distribution)
print(sum(state_distribution))
print(min(state_distribution))
j = molecule.state_df["j"].to_numpy()
print(j)
print(len(j))

[1.00000000e+00 1.00000000e+00 9.54972501e-01 ... 3.07860693e-26
 3.07860693e-26 3.07860693e-26]
5202
[1.14300086e-02 1.14300086e-02 1.09153439e-02 ... 3.51885037e-28
 3.51885037e-28 3.51885037e-28]
0.999999999999992
3.5188503689774155e-28
[ 0  0  1 ... 50 50 50]
5202


In [23]:
unique_values = np.unique(state_distribution)
print(unique_values)
print(len(unique_values))

[3.51885037e-28 3.52255268e-27 3.36748026e-26 3.07428048e-25
 2.68023443e-24 2.23147971e-23 1.77420570e-22 1.34711869e-21
 9.76784188e-21 6.76366744e-20 4.47256578e-19 2.82437328e-18
 1.70324968e-17 9.80901517e-17 5.39465110e-16 2.83329753e-15
 1.42105822e-14 6.80647767e-14 3.11332059e-13 1.35992867e-12
 5.67282362e-12 2.25981714e-11 8.59682691e-11 3.12315783e-10
 1.08352902e-09 3.58986429e-09 1.13581181e-08 3.43182863e-08
 9.90229074e-08 2.72857955e-07 7.18006593e-07 1.80430981e-06
 4.32996769e-06 9.92313880e-06 2.17172256e-05 4.53889865e-05
 9.05915092e-05 1.72669396e-04 3.14292567e-04 5.46315746e-04
 9.06868237e-04 1.43759161e-03 2.17629486e-03 3.14623246e-03
 4.34364921e-03 5.72676745e-03 7.21033126e-03 8.66945414e-03
 9.95449322e-03 1.09153439e-02 1.14300086e-02]
51


In [50]:
dephased = False
coherence_time_us = 1
is_minus = False
max_frequency_mhz = 1e-3
scan_points = 100

prob = np.dot(get_excitation_probabilities(molecule, frequency, duration_us, rabi_rate_mhz, dephased, coherence_time_us, is_minus), state_distribution)

print(prob)

frequencies = np.linspace(-max_frequency_mhz, max_frequency_mhz, scan_points)
exc_probs = [
    np.dot(
        get_excitation_probabilities(molecule, frequency, duration_us, rabi_rate_mhz, dephased, coherence_time_us, is_minus),
        state_distribution,
    )
    for frequency in frequencies
]
print(frequencies)
print(len(frequencies))

0.00893256434001637
[-1.00000000e-03 -9.79797980e-04 -9.59595960e-04 -9.39393939e-04
 -9.19191919e-04 -8.98989899e-04 -8.78787879e-04 -8.58585859e-04
 -8.38383838e-04 -8.18181818e-04 -7.97979798e-04 -7.77777778e-04
 -7.57575758e-04 -7.37373737e-04 -7.17171717e-04 -6.96969697e-04
 -6.76767677e-04 -6.56565657e-04 -6.36363636e-04 -6.16161616e-04
 -5.95959596e-04 -5.75757576e-04 -5.55555556e-04 -5.35353535e-04
 -5.15151515e-04 -4.94949495e-04 -4.74747475e-04 -4.54545455e-04
 -4.34343434e-04 -4.14141414e-04 -3.93939394e-04 -3.73737374e-04
 -3.53535354e-04 -3.33333333e-04 -3.13131313e-04 -2.92929293e-04
 -2.72727273e-04 -2.52525253e-04 -2.32323232e-04 -2.12121212e-04
 -1.91919192e-04 -1.71717172e-04 -1.51515152e-04 -1.31313131e-04
 -1.11111111e-04 -9.09090909e-05 -7.07070707e-05 -5.05050505e-05
 -3.03030303e-05 -1.01010101e-05  1.01010101e-05  3.03030303e-05
  5.05050505e-05  7.07070707e-05  9.09090909e-05  1.11111111e-04
  1.31313131e-04  1.51515152e-04  1.71717172e-04  1.91919192e-04
  2.1

In [3]:
import numpy as np
from molecules import Molecule
from scipy.constants import h, k
from scipy.sparse import csr_array, sparray
from typing import Tuple, Optional, NamedTuple
from wigners import clebsch_gordan


class Polarization(NamedTuple):
    pi: float
    sp: float
    sm: float


def get_excitation_probabilities(
    molecule: Molecule,
    frequency: float,
    duration_us: float,
    rabi_rate_mhz: float,
    dephased: bool = False,
    coherence_time_us: float = 100.0,
    is_minus: bool = True,
) -> np.ndarray:
    """Returns the excitation probabilities for given frequency and other parameters

    Args:
        molecule (Molecule): The molecule to calculate the excitation probabilities for
        frequency (float): The frequency of the excitation pulse in MHz
        duration_us (float): The duration of the excitation pulse in microseconds
        rabi_rate_mhz (float): The Rabi rate in MHz
        dephased (bool): If True, the excitation is dephased
        coherence_time_us (float): The coherence time in us for rabi flopping
        is_minus (bool): If True, the excitation is for dm = -1
    Returns:
        np.ndarray: The excitation probabilities for each state
    """
    state_exc_probs = np.zeros(len(molecule.state_df))      # number of rows

    detunings = 2 * np.pi * (frequency - molecule.transition_df["energy_diff"].to_numpy(dtype=float) * 1e-3)
    omegas = rabi_rate_mhz * molecule.transition_df["coupling"].to_numpy(dtype=float)

    if dephased:
        transition_exc_probs = omegas**2 / (omegas**2 + detunings**2) * ((1 - np.cos(np.sqrt(omegas**2.0 + detunings**2.0) * duration_us) * np.exp(-duration_us / coherence_time_us)) / 2)
    else:
        transition_exc_probs = omegas**2 / (omegas**2 + detunings**2) * np.sin(np.sqrt(omegas**2.0 + detunings**2.0) * duration_us / 2) ** 2
    if is_minus:
        # state1 --> state2
        states_index = molecule.transition_df["index1"].to_numpy(dtype=int)
    else:
        # state2 --> state1
        states_index = molecule.transition_df["index2"].to_numpy(dtype=int)
    for i in range(len(molecule.transition_df)):
        state_exc_probs[states_index[i]] += transition_exc_probs[i]

    # MAYBE IMPROVEMENTS: non si possono fare operazioni direttamente con i pandas df?
    return state_exc_probs


def get_thermal_distribution(molecule: Molecule, temperature: float) -> np.ndarray:
    """Returns the thermal distribution for a given temperature

    Args:
        molecule (Molecule): The molecule to calculate the thermal distribution for
        temperature (float): The temperature in Kelvin
    Returns:
        np.ndarray: The thermal distribution for each state
    """
    rotational_energy_ghz = molecule.state_df["rotation_energy_ghz"].to_numpy()
    state_distribution = np.exp(-h * rotational_energy_ghz * 1e9 / (k * temperature))

    state_distribution /= np.sum(state_distribution)
    return state_distribution


def get_spectrum(
    molecule: Molecule,
    state_distribution: np.ndarray,
    duration_us: float,
    rabi_rate_mhz: float,
    max_frequency_mhz: float,
    scan_points: int,
    dephased: bool = True,
    coherence_time_us: float = 100.0,
    is_minus: bool = True,
) -> Tuple[np.ndarray, np.ndarray]:
    """Returns the spectrum for given parameters

    Args:
        molecule (Molecule): The molecule to calculate the spectrum for
        duration_us (float): The duration of the excitation pulse in microseconds
        rabi_rate_mhz (float): The Rabi rate in MHz
        max_frequency_mhz (float): The maximum frequency in MHz
        scan_points (int): The number of scan points
        dephased (bool): If True, the excitation is dephased
        is_minus (bool): If True, the excitation is for dm = -1; otherwise, dm = +1
    Returns:
        np.ndarray: The excitation probabilities for each frequency
    """
    frequencies = np.linspace(-max_frequency_mhz, max_frequency_mhz, scan_points)
    exc_probs = [
        np.dot(
            get_excitation_probabilities(molecule, frequency, duration_us, rabi_rate_mhz, dephased, coherence_time_us, is_minus),
            state_distribution,
        )
        for frequency in frequencies
    ]
    return frequencies, exc_probs


def excitation_matrix(
    molecule: Molecule,
    frequency: float,
    duration_us: float,
    rabi_rate_mhz: float,
    dephased: bool = False,
    coherence_time_us: float = 1000.0,
    is_minus: bool = True,
) -> sparray:
    """Returns the excitation probabilities for given frequency and other parameters

    Args:
        molecule (Molecule): The molecule to calculate the excitation probabilities for
        frequency (float): The frequency of the excitation pulse in MHz
        duration_us (float): The duration of the excitation pulse in microseconds
        rabi_rate_mhz (float): The Rabi rate in MHz
        dephased (bool): If True, the excitation is dephased
        coherence_time_us (float): The coherence time in us for rabi flopping
        is_minus (bool): If True, the excitation is for dm = -1
    Returns:
        np.ndarray: The excitation probabilities for each state
    """
    num_states = len(molecule.state_df)
    detunings = 2 * np.pi * (frequency - molecule.transition_df["energy_diff"].to_numpy(dtype=float) * 1e-3)
    omegas = rabi_rate_mhz * molecule.transition_df["coupling"].to_numpy(dtype=float)
    if dephased:
        transition_exc_probs = omegas**2 / (omegas**2 + detunings**2) * ((1 - np.cos(np.sqrt(omegas**2.0 + detunings**2.0) * duration_us) * np.exp(-duration_us / coherence_time_us)) / 2)
    else:
        transition_exc_probs = omegas**2 / (omegas**2 + detunings**2) * np.sin(np.sqrt(omegas**2.0 + detunings**2.0) * duration_us / 2) ** 2
    if is_minus:
        # state1 --> state2
        rows = molecule.transition_df["index2"].to_numpy(dtype=int)
        cols = molecule.transition_df["index1"].to_numpy(dtype=int)
        exc_matrix = csr_array((transition_exc_probs, (rows, cols)), shape=(num_states, num_states)) + csr_array((-transition_exc_probs, (cols, cols)), shape=(num_states, num_states))
    else:
        # state2 --> state1
        rows = molecule.transition_df["index1"].to_numpy(dtype=int)
        cols = molecule.transition_df["index2"].to_numpy(dtype=int)
        exc_matrix = csr_array((transition_exc_probs, (rows, cols)), shape=(num_states, num_states)) + csr_array((-transition_exc_probs, (cols, cols)), shape=(num_states, num_states))

    return exc_matrix


def get_ac_stark_shifts(molecule: Molecule, rabi_rate_mhz_1: float, rabi_rate_mhz_2: float, q1: Polarization, q2: Polarization) -> np.ndarray:
    """Returns the AC Stark shifts for the given Rabi rates and q0 values

    Args:
        rabi_rate_mhz_1 (float): The Rabi rate for the first beam in MHz
        rabi_rate_mhz_2 (float): The Rabi rate for the second beam in MHz
        q1 (Polarization): Polarization components of the first beam
        q2 (Polarization): Polarization components of the second beam
    Returns:
        np.ndarray: The AC Stark shifts for each state
    """

    def coupling(j, j_exc, mj, q):
        if j < abs(mj) or j_exc < abs(mj + q):
            return 0
        return clebsch_gordan(1, 0, j_exc, 0, j, 0) * clebsch_gordan(1, -q, j_exc, mj + q, j, mj) * clebsch_gordan(1, 0, j_exc, 0, j, 0) * clebsch_gordan(1, -q, j_exc, mj + q, j, mj)
        # return clebsch_gordan(1, 0, j_exc, 0, j, 0) * clebsch_gordan(1, -q, j_exc, mj + q, j, mj) * clebsch_gordan(1, 0, j, 0, j_exc, 0) * clebsch_gordan(1, q, j, mj, j_exc, mj + q)

    num_states = len(molecule.state_df)
    states_array = molecule.state_df.to_numpy()

    ac_stark_shifts = np.zeros(num_states)

    for i in range(num_states):
        j = states_array[i, 0]
        m = states_array[i, 1]
        mj_up = int(m - 0.5)
        mj_down = int(m + 0.5)
        spin_up = states_array[i, 3]
        spin_down = states_array[i, 4]
        ac_stark_shifts_1 = molecule.coupling_coefficient * (
            spin_down**2
            * (
                q1.pi**2 * (coupling(j, j + 1, mj_down, 0) + coupling(j, j - 1, mj_down, 0))
                + q1.sp**2 * (coupling(j, j + 1, mj_down, 1) + coupling(j, j - 1, mj_down, 1))
                + q1.sm**2 * (coupling(j, j + 1, mj_down, -1) + coupling(j, j - 1, mj_down, -1))
            )
            + spin_up**2
            * (
                q1.pi**2 * (coupling(j, j + 1, mj_up, 0) + coupling(j, j - 1, mj_up, 0))
                + q1.sp**2 * (coupling(j, j + 1, mj_up, 1) + coupling(j, j - 1, mj_up, 1))
                + q1.sm**2 * (coupling(j, j + 1, mj_up, -1) + coupling(j, j - 1, mj_up, -1))
            )
        )
        ac_stark_shifts_1 *= rabi_rate_mhz_1

        ac_stark_shifts_2 = molecule.coupling_coefficient * (
            spin_down**2
            * (
                q2.pi**2 * (coupling(j, j + 1, mj_down, 0) + coupling(j, j - 1, mj_down, 0))
                + q2.sp**2 * (coupling(j, j + 1, mj_down, 1) + coupling(j, j - 1, mj_down, 1))
                + q2.sm**2 * (coupling(j, j + 1, mj_down, -1) + coupling(j, j - 1, mj_down, -1))
            )
            + spin_up**2
            * (
                q2.pi**2 * (coupling(j, j + 1, mj_up, 0) + coupling(j, j - 1, mj_up, 0))
                + q2.sp**2 * (coupling(j, j + 1, mj_up, 1) + coupling(j, j - 1, mj_up, 1))
                + q2.sm**2 * (coupling(j, j + 1, mj_up, -1) + coupling(j, j - 1, mj_up, -1))
            )
        )
        ac_stark_shifts_2 *= rabi_rate_mhz_2
        ac_stark_shifts[i] = ac_stark_shifts_1 + ac_stark_shifts_2

    return ac_stark_shifts


class States:
    """Class to store the state distribution of the given molecule"""

    def __init__(self, molecule: Molecule, temperature: Optional[float] = None):
        """Initializes the States object

        Args:
            molecule (Molecule): The molecule to store the state distribution for
            temperature (float): The temperature in Kelvin, if None, the state distribution is uniform
        """
        self.molecule = molecule
        self.num_states = len(molecule.state_df)
        self.j = molecule.state_df["j"].to_numpy(dtype=int)

        if temperature is not None:
            self.dist = get_thermal_distribution(molecule, temperature)
        else:
            self.dist = np.ones(len(molecule.state_df)) / len(molecule.state_df)

    def j_distribution(self) -> np.ndarray:
        """Returns the distribution of the rotational states"""
        return np.bincount(self.j, weights=self.dist)
